In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import random as rnd
import os
import io
import math
from tqdm import tqdm
import csv
import json
import gc
from pathlib import Path


from sklearn.preprocessing import StandardScaler, RobustScaler

import cv2
import mediapipe as mp
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.distributions import Categorical
# from torch.nn.functional import relu, softmax   



In [70]:
class MediapipeSegmentationService:
    """
    MediaPipe Pose → normalized 3D pose pipeline with correct visualization.
    """

    def __init__(self):
        self.mp_pose = mp.solutions.pose

        # Selected MediaPipe joint indices (13 joints)
        self.joint_ids = [
            0,    # nose
            11, 12,  # shoulders
            13, 14,  # elbows
            15, 16,  # wrists
            23, 24,  # hips
            25, 26,  # knees
            27, 28   # ankles
        ]

    # ------------------------------------------------------------ #
    # Core pose normalization
    # ------------------------------------------------------------ #

    def _normalize_pose(self, landmarks, frame_w, frame_h):
        """
        Returns:
            pose_3d   : (13, 3) normalized person-centric
            reference : (2,) mid-hip in pixels
            bbox_size : (2,)
            min_xy    : (2,)
        """

        if landmarks is None:
            return None, None, None, None

        # Convert to pixel coordinates
        coords = np.array([
            [
                landmarks.landmark[i].x * frame_w,
                landmarks.landmark[i].y * frame_h,
                landmarks.landmark[i].z
            ]
            for i in self.joint_ids
        ])

        # Mid-hip reference
        left_hip, right_hip = coords[7], coords[8]
        reference = (left_hip + right_hip)[:2] / 2

        coords_shifted = coords.copy()
        coords_shifted[:, :2] -= reference

        # Bounding box
        min_xy = coords_shifted[:, :2].min(axis=0)
        max_xy = coords_shifted[:, :2].max(axis=0)
        bbox_size = max_xy - min_xy
        bbox_size[bbox_size == 0] = 1.0

        # Normalize to [0,1]
        pose_3d = coords_shifted.copy()
        pose_3d[:, 0] = (coords_shifted[:, 0] - min_xy[0]) / bbox_size[0]
        pose_3d[:, 1] = (coords_shifted[:, 1] - min_xy[1]) / bbox_size[1]

        # Scale normalize depth (optional but recommended)
        pose_3d[:, 2] /= np.linalg.norm(bbox_size)

        return pose_3d, reference, bbox_size, min_xy

    # ------------------------------------------------------------ #
    # Video processing
    # ------------------------------------------------------------ #

    def process_video(self, video_path):
        """
        Full pipeline:
        video → list of frame dicts + (T, 13, 3) tensor
        """

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise IOError(f"Cannot open video: {video_path}")

        frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        frames = []
        poses_3d = []

        with self.mp_pose.Pose(
            model_complexity=2,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        ) as pose:
            
            i = 0
            while cap.isOpened():
                print(i, end='\r')
                i += 1
                success, frame = cap.read()
                if not success:
                    break

                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                result = pose.process(frame_rgb)

                pose_3d, ref, bbox, min_xy = self._normalize_pose(
                    result.pose_landmarks,
                    frame_w,
                    frame_h
                )

                frames.append({
                    "pose_3d": pose_3d,
                    "reference": ref,
                    "bbox_size": bbox,
                    "min_xy": min_xy
                })

                poses_3d.append(
                    pose_3d if pose_3d is not None else None
                )

        cap.release()

        tensor_3d, mask = self._stack_tensor(poses_3d)

        return frames, tensor_3d, mask

    # ------------------------------------------------------------ #
    # Tensor stacking
    # ------------------------------------------------------------ #

    def _stack_tensor(self, poses):
        valid = [p for p in poses if p is not None]
        T = len(poses)
        J = valid[0].shape[0]

        tensor = np.zeros((T, J, 3))
        mask = np.zeros(T, dtype=bool)

        for t, p in enumerate(poses):
            if p is None:
                continue
            tensor[t] = p
            mask[t] = True

        return tensor, mask

    def process_video_cosine_segments(self, video_path):
        """
        Returns:
            cos_tensor: (T, num_segments)
            mask      : (T,)
        """

        frames, tensor_3d, mask = self.process_video(video_path)

        # Segment definitions (joint indices)
        segments = [
            (1, 3),   # L shoulder → elbow
            (3, 5),   # L elbow → wrist
            (2, 4),   # R shoulder → elbow
            (4, 6),   # R elbow → wrist
            (7, 9),   # L hip → knee
            (9, 11),  # L knee → ankle
            (8, 10),  # R hip → knee
            (10, 12), # R knee → ankle
            (1, 2), # Shoulders
            (7, 8)  # Hips
        ]

        T = tensor_3d.shape[0]
        S = len(segments)

        cos_tensor = np.zeros((T, S))

        for t in range(T):
            if not mask[t]:
                continue

            pose = tensor_3d[t]

            # --- Hip & shoulder midpoints ---
            hip_mid = (pose[7] + pose[8]) / 2
            shoulder_mid = (pose[1] + pose[2]) / 2

            torso_vec = shoulder_mid - hip_mid

            # --- Enforce consistent "up" direction ---
            if torso_vec[1] > 0:
                torso_vec = -torso_vec

            torso_norm = np.linalg.norm(torso_vec)
            if torso_norm == 0:
                continue

            torso_vec /= torso_norm

            # --- Segment cosine similarities ---
            for i, (a, b) in enumerate(segments):
                seg_vec = pose[b] - pose[a]
                seg_norm = np.linalg.norm(seg_vec)

                if seg_norm == 0:
                    cos_tensor[t, i] = 0.0
                    continue

                seg_vec /= seg_norm
                cos_tensor[t, i] = np.dot(seg_vec, torso_vec)

        return cos_tensor, mask
    # ------------------------------------------------------------ #
    # Visualization
    # ------------------------------------------------------------ #

    
    def save_visualized_video(self, output_path, frames, video_path=None, pose_labels=None):
        """
        Reprojects normalized pose back to image space.
        If video_path is None, renders pose on a black background.
        """

        connections = [
            (0,1),(0,2),
            (1,2),
            (1,3),(2,4),
            (3,5),(4,6),
            (1,7),(2,8),
            (7,8),
            (7,9),(8,10),
            (9,11),(10,12)
        ]

        # -------------------------------------------------- #
        # Case 1: No video → black background
        # -------------------------------------------------- #
        if not video_path:
            # Choose a reasonable canvas size
            w, h = 800, 800
            fps = 30

            out = cv2.VideoWriter(
                output_path,
                cv2.VideoWriter_fourcc(*"mp4v"),
                fps,
                (w, h)
            )

            for idx, data in enumerate(frames):
                frame = np.zeros((h, w, 3), dtype=np.uint8)
                pose = data["pose_3d"]

                if pose is not None:
                    px = pose.copy()

                    # Center on screen using bbox size
                    px[:, 0] = pose[:, 0] * data["bbox_size"][0] + w // 2
                    px[:, 1] = pose[:, 1] * data["bbox_size"][1] + h // 2

                    px = px[:, :2].astype(int)

                    for x, y in px:
                        cv2.circle(frame, (x, y), 6, (0, 255, 0), -1)

                    for s, e in connections:
                        cv2.line(frame, tuple(px[s]), tuple(px[e]), (200, 200, 200), 4)
                        
                    if pose_labels is not None and pose_labels[idx][0] is not None:
                        cv2.putText(
                            frame,
                            str(pose_labels[idx]),
                            org=(150, 150),
                            fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                            fontScale=1,
                            color=(0, 180, 255),
                            thickness=3,
                            lineType=cv2.LINE_AA,
                        )
                        
                out.write(frame)

            out.release()
            print(f"Saved visualization to {output_path}")
            return

        # -------------------------------------------------- #
        # Case 2: Overlay on original video
        # -------------------------------------------------- #
        cap = cv2.VideoCapture(video_path)
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        out = cv2.VideoWriter(
            output_path,
            cv2.VideoWriter_fourcc(*"mp4v"),
            fps,
            (w, h)
        )

        idx = 0
        while cap.isOpened() and idx < len(frames):
            success, frame = cap.read()
            if not success:
                break

            data = frames[idx]
            pose = data["pose_3d"]

            if pose is not None:
                px = pose.copy()

                px[:, 0] = (
                    pose[:, 0] * data["bbox_size"][0]
                    + data["min_xy"][0]
                    + data["reference"][0]
                )

                px[:, 1] = (
                    pose[:, 1] * data["bbox_size"][1]
                    + data["min_xy"][1]
                    + data["reference"][1]
                )

                px = px[:, :2].astype(int)

                for x, y in px:
                    cv2.circle(frame, (x, y), 6, (200, 200, 200), -1)

                for s, e in connections:
                    cv2.line(frame, tuple(px[s]), tuple(px[e]), (0, 255, 0), 4)
                    
                if pose_labels is not None and pose_labels[idx][0] is not None:
                    cv2.putText(
                        frame,
                        str(pose_labels[idx]),
                        org=(150, 150),
                        fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                        fontScale=1,
                        color=(0, 180, 255),
                        thickness=3,
                        lineType=cv2.LINE_AA,
                    )

            out.write(frame)
            idx += 1

        cap.release()
        out.release()
        print(f"Saved visualization to {output_path}")

In [71]:
segmentation_service = MediapipeSegmentationService()

In [16]:
# Extracting datapoints example
video_path = "../data\exc0\IMG_h3.MOV"
segmentor = MediapipeSegmentationService()

tensor_3d, mask = segmentor.process_video_cosine_segments(video_path)

print(f"landmarks:\n{tensor_3d[0]}")

landmarks:
[-0.88224355 -0.74343301 -0.78846572 -0.64818296 -0.70738226  0.76864476
 -0.98790701 -0.09628052  0.41661257  0.36135471]


In [38]:
class AnalyticalModel():
    
    def __init__(self, reference_json):
        self.reference_json = reference_json
        self.segmentor = MediapipeSegmentationService()
        
        self.video_extentions = [".mp4", ".mov", ".avi", ".mkv", ".wmv", ".flv", ".webm"]
    
    def _get_video_files(self, folder):
        folder = Path(folder)
        return [
            str(p)
            for p in folder.iterdir()
            if p.is_file() and p.suffix.lower() in self.video_extentions
        ]
        
    @staticmethod
    def _similarity_score(x, mean, std):
        x = np.asarray(x)
        mean = np.asarray(mean)
        std = np.asarray(std)

        z = (x - mean) / (std + 1e-8)
        dist = np.mean(np.abs(z))
        return float(np.exp(-dist))

    def fit(self, pose_name, video_reference_folder):
        videos = self._get_video_files(video_reference_folder)
        segments_bag = []
        for video in videos:
            segment_cos_distances, _ = segmentor.process_video_cosine_segments(video)
            segments_bag.append(segment_cos_distances)
        
        segments_bag = np.vstack(segments_bag)
        pose_mean = np.mean(segments_bag, axis=0)  # (S,)
        pose_std  = np.std(segments_bag, axis=0) 
        
        
        with open(file=self.reference_json, mode='r') as file:
            reference = json.load(file)
            
            reference[pose_name]= {
                "mean": pose_mean.tolist(),
                "std": pose_std.tolist()
            }
        
        with open(file=self.reference_json, mode='w') as file:
            json.dump(reference, file, indent=2)
    
    def get_poses_from_video(self, video_path):

        with open(self.reference_json, mode="r") as file:
            reference = json.load(file)

        segment_cos_distances, _ = self.segmentor.process_video_cosine_segments(video_path)

        frame_results = []

        for cos_distance in segment_cos_distances:
            similarities = {}

            for pose_name, normal_position in reference.items():

                similarity_score = AnalyticalModel._similarity_score(
                    cos_distance,
                    normal_position["mean"],
                    normal_position["std"]
                )

                similarities[pose_name] = similarity_score

            if similarities:
                best_pose = max(similarities, key=similarities.get)
                best_score = similarities[best_pose]
            else:
                best_pose = None
                best_score = 0.0

            frame_results.append((best_pose, float(best_score)))

        return frame_results
            

In [28]:
analytical_model = AnalyticalModel(reference_json="../data/pose_reference_points/reference.json")
dataset = [
    ('starting', '..\data\pose_reference_points\starting'),
    ('swing', '..\data\pose_reference_points\swing'),
    ('landing', '..\data\pose_reference_points\landing')
]
for pose_name, video_reference_folder in dataset:
    analytical_model.fit(pose_name=pose_name, video_reference_folder=video_reference_folder)

In [42]:
# Extracting datapoints example
analytical_model = AnalyticalModel(reference_json="../data/pose_reference_points/reference.json")

video_path = "../data\exc0\IMG_h15.MOV"
segmentor = MediapipeSegmentationService()

frames, tensor_3d, mask = segmentor.process_video(video_path)
poses = analytical_model.get_poses_from_video(video_path)

poses = [f"{pose}: {score:0.4f}" for pose, score in poses]

print(f"Poses:\n{poses}")

Poses:
['landing: 0.5041', 'landing: 0.5304', 'landing: 0.5376', 'landing: 0.5056', 'landing: 0.4976', 'landing: 0.4261', 'starting: 0.3453', 'starting: 0.3842', 'starting: 0.4013', 'starting: 0.4541', 'starting: 0.4818', 'starting: 0.5152', 'starting: 0.5166', 'starting: 0.5137', 'starting: 0.5120', 'starting: 0.5201', 'starting: 0.5149', 'starting: 0.4948', 'starting: 0.5089', 'starting: 0.5339', 'starting: 0.5648', 'starting: 0.5896', 'starting: 0.6325', 'starting: 0.6714', 'starting: 0.7219', 'starting: 0.7260', 'starting: 0.7407', 'starting: 0.6819', 'starting: 0.6460', 'starting: 0.6903', 'starting: 0.6615', 'starting: 0.6678', 'starting: 0.5830', 'starting: 0.5656', 'starting: 0.5784', 'starting: 0.6241', 'starting: 0.5491', 'starting: 0.4939', 'starting: 0.4966', 'starting: 0.4589', 'starting: 0.4155', 'starting: 0.4329', 'starting: 0.3875', 'starting: 0.3593', 'starting: 0.3575', 'landing: 0.3205', 'starting: 0.3345', 'starting: 0.3989', 'starting: 0.4078', 'starting: 0.3953',

In [73]:
segmentation_service.save_visualized_video("labeled_empty.mp4", frames, pose_labels=poses)

Saved visualization to labeled_empty.mp4
